# AI1: Evaluation-Only Diagnostics for Saved Checkpoints (Self-Contained)

### Post-Hoc Diagnostics Notice
- **NO Retraining**: Models evaluated strictly in read-only inference mode with zero parameter updates.
- **NO Reselection**: All test evaluations are diagnostic; original validation selection remains frozen.
- **Disambiguated Candidates**: Unique evidence identities `candidate_id::evidence_id::record_id` resolve all candidate collisions.
- **Session Recovery**: Accepts either existing `/kaggle/working/experiment_outputs` or attached private results ZIP (`EXPERIMENT_ROOT` / `RESULTS_ZIP`).
- **Self-Contained Trusted Code**: Embedded helper code is written to `/kaggle/working/runner` with no external script downloads.

In [ ]:
# 1. Environment & Path Discovery
import os, sys, json, math, shutil, hashlib, platform, subprocess, zipfile
from pathlib import Path

WORKING_DIR = Path('/kaggle/working').resolve() if Path('/kaggle/working').exists() else Path('./run_working').resolve()
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Discover Bundle Root
def discover_bundle_root():
    env_root = os.environ.get('BUNDLE_ROOT')
    if env_root and Path(env_root).exists() and (Path(env_root) / 'manifest.json').exists():
        return Path(env_root).resolve()
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates = [p for p in kaggle_input.rglob('manifest.json') if p.is_file()]
        if len(candidates) == 1:
            return candidates[0].parent.resolve()
        elif len(candidates) > 1:
            raise ValueError(f'Multiple manifests in /kaggle/input: {candidates}')
    for candidate_name in ['ai1-kaggle-silver-v2', 'ai1-kaggle-silver-v1']:
        local_cand = Path(f'./artifacts/{candidate_name}').resolve()
        if local_cand.exists() and (local_cand / 'manifest.json').exists():
            return local_cand
    raise FileNotFoundError('Valid bundle root containing manifest.json not found. Set BUNDLE_ROOT.')

BUNDLE_DIR = discover_bundle_root()
print(f'Bundle root: {BUNDLE_DIR}')


In [ ]:
# 2. Safe Session Recovery & Experiment Root Discovery
# Supports either:
# A) Existing /kaggle/working/experiment_outputs
# B) Configurable EXPERIMENT_ROOT env var
# C) Attached private dataset containing experiment results zip (or zip in /kaggle/input)

def resolve_experiment_root():
    # Check explicit env var first
    env_exp = os.environ.get('EXPERIMENT_ROOT')
    if env_exp:
        p = Path(env_exp).resolve()
        if p.exists() and (p / 'experiment_results.json').exists():
            return p, None
        elif p.is_file() and zipfile.is_zipfile(p):
            return None, p

    # Check default working directory
    default_exp = WORKING_DIR / 'experiment_outputs'
    if default_exp.exists() and (default_exp / 'experiment_results.json').exists():
        return default_exp, None

    # Search for results zip in /kaggle/input or working dir
    search_roots = [Path('/kaggle/input'), WORKING_DIR]
    candidate_zips = []
    for s_root in search_roots:
        if s_root.exists():
            for z in s_root.rglob('*.zip'):
                if 'ai1' in z.name.lower() or 'results' in z.name.lower() or 'experiment' in z.name.lower():
                    candidate_zips.append(z)

    if len(candidate_zips) == 1:
        return None, candidate_zips[0].resolve()
    elif len(candidate_zips) > 1:
        # Choose one containing experiment_results.json
        for z in candidate_zips:
            try:
                with zipfile.ZipFile(z, 'r') as zf:
                    if 'experiment_results.json' in zf.namelist():
                        return None, z.resolve()
            except Exception:
                continue

    raise FileNotFoundError(
        'Could not locate experiment outputs or results archive. '
        'Please set EXPERIMENT_ROOT environment variable or attach outputs ZIP.'
    )

EXP_DIR, EXP_ZIP = resolve_experiment_root()
print(f'Experiment Directory: {EXP_DIR}')
print(f'Experiment ZIP: {EXP_ZIP}')


In [ ]:
# 3. Write Embedded Trusted Evaluation Helper to Separate Code Dir
RUNNER_DIR = WORKING_DIR / 'runner'
RUNNER_DIR.mkdir(parents=True, exist_ok=True)
EVAL_SCRIPT_PATH = RUNNER_DIR / 'evaluate_ai1_saved_checkpoints.py'

# Write verbatim verified runner source
EVAL_SCRIPT_SOURCE = "#!/usr/bin/env python3\n\"\"\"Evaluation-only script for existing AI1 Kaggle checkpoints without retraining.\n\nStrict evaluation-only runner:\n- Reuses existing trained checkpoints from experiment_results.json and checkpoint dir\n- Evaluates Arms A (baseline step 0), B (selected step from summary/dir), C (selected step from summary/dir)\n- Evaluates the same frozen restricted candidate pool across all arms\n- Resolves candidate ambiguity via unique evidence identity (candidate_id::evidence_id::record_id)\n- Computes micro and macro_job MRR, Hit@1, Hit@3, Recall@1, Recall@3 (all queries, mixed-only, positive-only, no-positive)\n- Computes pairwise positive-vs-negative ordering and tracks tie counts\n- Computes paired JD bootstrap deltas C - A and C - B\n- Exports per-arm per-query candidate ranking with full provenance and file hashes\n- Explicitly labels all test analyses as post-hoc diagnostics, NOT selection criteria\n- Fails closed on path traversal, symlinks, missing checkpoints (no fallback), non-finite scores, or output collisions\n\"\"\"\n\nimport argparse\nimport copy\nfrom collections import defaultdict\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport re\nimport sys\nimport unicodedata\nimport zipfile\nfrom typing import Any, Dict, List, Optional, Set, Tuple\n\n\n# ==============================================================================\n# Custom Exceptions (Fail-closed)\n# ==============================================================================\n\nclass RefuseOverwriteError(Exception):\n    \"\"\"Raised when output directory exists and is not empty.\"\"\"\n    pass\n\n\nclass SmokeExperimentError(Exception):\n    \"\"\"Raised when experiment_results.json represents a smoke/synthetic run.\"\"\"\n    pass\n\n\nclass MissingCheckpointError(Exception):\n    \"\"\"Raised when a selected checkpoint directory does not exist on disk.\"\"\"\n    pass\n\n\nclass PathTraversalError(Exception):\n    \"\"\"Raised when a path traverses outside allowed boundaries or contains symlinks.\"\"\"\n    pass\n\n\nclass NonFiniteScoreError(Exception):\n    \"\"\"Raised when cosine similarity calculation results in non-finite value.\"\"\"\n    pass\n\n\nclass CheckpointValidationError(Exception):\n    \"\"\"Raised when checkpoint metadata or layout violates strict selection rules.\"\"\"\n    pass\n\n\nclass SafeArchiveError(Exception):\n    \"\"\"Raised when archive extraction violates strict safety rules.\"\"\"\n    pass\n\n\n# ==============================================================================\n# Safe Archive Extraction & Recovery\n# ==============================================================================\n\nALLOWED_ARCHIVE_EXTENSIONS = {\n    \".json\", \".jsonl\", \".safetensors\", \".txt\", \".bin\", \".pt\", \".pth\", \".md\"\n}\n\nALLOWED_ARCHIVE_NAMES = {\n    \"experiment_results.json\",\n    \"test_predictions.jsonl\",\n    \"config.json\",\n    \"model.safetensors\",\n    \"tokenizer.json\",\n    \"tokenizer_config.json\",\n    \"special_tokens_map.json\",\n    \"vocab.txt\",\n    \"sentence_bert_config.json\",\n    \"modules.json\",\n    \"config_sentence_transformers.json\",\n}\n\nMAX_ARCHIVE_ENTRIES = 1000\nMAX_ENTRY_SIZE_BYTES = 500 * 1024 * 1024  # 500 MiB for model weights\nMAX_TOTAL_EXTRACTED_BYTES = 2 * 1024 * 1024 * 1024  # 2 GiB\n\n\ndef safe_extract_results_archive(\n    archive_path: Path | str,\n    target_dir: Path | str,\n) -> Path:\n    \"\"\"Safely extract results archive into target_dir with strict safety guarantees.\n\n    Guarantees:\n    - target_dir must be a fresh empty directory\n    - Rejects absolute paths, '..', backslashes, symlinks, duplicate member names\n    - Enforces allowlist on file extensions/names\n    - Enforces entry count and size limits\n    - Never executes any code from archive\n    \"\"\"\n    arc = Path(archive_path).resolve()\n    if not arc.exists() or not arc.is_file():\n        raise FileNotFoundError(f\"Archive file not found: {arc}\")\n\n    out_p = Path(target_dir).resolve()\n    if out_p.exists() and any(out_p.iterdir()):\n        raise SafeArchiveError(f\"Target directory for archive extraction is not empty: {out_p}\")\n    out_p.mkdir(parents=True, exist_ok=True)\n\n    if not zipfile.is_zipfile(arc):\n        raise SafeArchiveError(f\"File is not a valid zip archive: {arc}\")\n\n    with zipfile.ZipFile(arc, \"r\") as zf:\n        infolist = zf.infolist()\n        if len(infolist) > MAX_ARCHIVE_ENTRIES:\n            raise SafeArchiveError(f\"Archive exceeds maximum entries limit: {len(infolist)} > {MAX_ARCHIVE_ENTRIES}\")\n\n        seen_names: Set[str] = set()\n        total_extracted_bytes = 0\n\n        for member in infolist:\n            name = member.filename\n\n            # Reject backslashes\n            if \"\\\\\" in name:\n                raise SafeArchiveError(f\"Rejected archive member with backslash: {name}\")\n\n            # Reject absolute paths or path traversal\n            member_path = Path(name)\n            if member_path.is_absolute() or \"..\" in member_path.parts:\n                raise SafeArchiveError(f\"Path traversal detected in archive member: {name}\")\n\n            # Reject duplicates\n            norm_name = str(member_path)\n            if norm_name in seen_names:\n                raise SafeArchiveError(f\"Duplicate archive member detected: {name}\")\n            seen_names.add(norm_name)\n\n            # Check symlink attributes\n            # In zip format, symlinks have mode attribute with S_IFLNK (0o120000)\n            mode = member.external_attr >> 16\n            if mode & 0o170000 == 0o120000:\n                raise SafeArchiveError(f\"Symlink detected in archive member: {name}\")\n\n            if member.is_dir():\n                continue\n\n            # Check entry size\n            if member.file_size > MAX_ENTRY_SIZE_BYTES:\n                raise SafeArchiveError(\n                    f\"Member size exceeds limit: {member.file_size} > {MAX_ENTRY_SIZE_BYTES} ({name})\"\n                )\n            total_extracted_bytes += member.file_size\n            if total_extracted_bytes > MAX_TOTAL_EXTRACTED_BYTES:\n                raise SafeArchiveError(\n                    f\"Total uncompressed size exceeds limit: {total_extracted_bytes} > {MAX_TOTAL_EXTRACTED_BYTES}\"\n                )\n\n            # Check allowlist\n            suffix = member_path.suffix.lower()\n            if suffix not in ALLOWED_ARCHIVE_EXTENSIONS:\n                raise SafeArchiveError(\n                    f\"Archive member '{name}' has disallowed extension '{suffix}'\"\n                )\n\n            dest_file = (out_p / member_path).resolve()\n            try:\n                dest_file.relative_to(out_p)\n            except ValueError:\n                raise SafeArchiveError(f\"Archive member extracts outside target directory: {dest_file}\")\n\n            # Double check destination isn't a symlink\n            if dest_file.is_symlink():\n                raise SafeArchiveError(f\"Destination path is a symlink: {dest_file}\")\n\n            dest_file.parent.mkdir(parents=True, exist_ok=True)\n            with zf.open(member) as src, open(dest_file, \"wb\") as dst:\n                while True:\n                    chunk = src.read(1024 * 1024)\n                    if not chunk:\n                        break\n                    dst.write(chunk)\n\n    return out_p\n\n\n# ==============================================================================\n# Hashing & File Integrity\n# ==============================================================================\n\ndef compute_sha256(filepath: Path | str) -> str:\n    \"\"\"Compute sha256 hex digest of a file in streaming chunks. Rejects symlinks.\"\"\"\n    p = Path(filepath).resolve()\n    if p.is_symlink():\n        raise PathTraversalError(f\"Refusing to compute SHA-256 on symlink: {p}\")\n    h = hashlib.sha256()\n    with open(p, \"rb\") as f:\n        while True:\n            chunk = f.read(1024 * 1024)\n            if not chunk:\n                break\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef directory_sha256_manifest(dirpath: Path | str) -> Dict[str, str]:\n    \"\"\"Compute relative path -> sha256 mapping for all regular files in a directory.\n\n    Rejects symlinks anywhere in directory tree.\n    \"\"\"\n    p = Path(dirpath).resolve()\n    manifest: Dict[str, str] = {}\n    if not p.is_dir():\n        return manifest\n    for file_p in sorted(p.rglob(\"*\")):\n        if file_p.is_symlink():\n            raise PathTraversalError(f\"Symlink rejected in directory tree: {file_p}\")\n        if file_p.is_file():\n            rel_str = str(file_p.relative_to(p))\n            manifest[rel_str] = compute_sha256(file_p)\n    return manifest\n\n\ndef verify_bundle_manifest(bundle_root: Path | str) -> Dict[str, Any]:\n    \"\"\"Verify all artifacts in bundle manifest.json. Fails closed on mismatch, symlink, or unexpected file.\"\"\"\n    root = Path(bundle_root).resolve()\n    manifest_path = root / \"manifest.json\"\n    if not manifest_path.exists():\n        raise FileNotFoundError(f\"Missing manifest.json in bundle root: {root}\")\n    if manifest_path.is_symlink():\n        raise PathTraversalError(f\"manifest.json cannot be a symlink: {manifest_path}\")\n\n    with open(manifest_path, \"r\", encoding=\"utf-8\") as f:\n        manifest = json.load(f)\n\n    artifacts = manifest.get(\"artifacts\", {})\n    if not artifacts:\n        raise ValueError(\"Empty or invalid artifacts in manifest.json\")\n\n    for rel_path, expected_sha in artifacts.items():\n        # Validate rel_path structure: no leading slash, no '..', no backslash\n        if \"\\\\\" in rel_path:\n            raise PathTraversalError(f\"Backslash in manifest path: {rel_path}\")\n        p_rel = Path(rel_path)\n        if p_rel.is_absolute() or \"..\" in p_rel.parts:\n            raise PathTraversalError(f\"Path traversal in manifest artifact: {rel_path}\")\n\n        artifact_path = (root / p_rel).resolve()\n        try:\n            artifact_path.relative_to(root)\n        except ValueError:\n            raise PathTraversalError(f\"Manifest artifact traverses outside bundle root: {rel_path}\")\n\n        if not artifact_path.exists():\n            raise FileNotFoundError(f\"Artifact listed in manifest missing on disk: {rel_path}\")\n        if artifact_path.is_symlink():\n            raise PathTraversalError(f\"Symlink rejected for manifest artifact: {rel_path}\")\n\n        actual_sha = compute_sha256(artifact_path)\n        if actual_sha != expected_sha:\n            raise ValueError(f\"SHA-256 mismatch for artifact '{rel_path}': expected {expected_sha}, got {actual_sha}\")\n\n    for p in root.rglob(\"*\"):\n        if p.is_symlink():\n            raise PathTraversalError(f\"Symlink detected in bundle directory: {p}\")\n        if p.is_file():\n            rel = str(p.relative_to(root))\n            if rel == \"manifest.json\" or rel.startswith(\".\"):\n                continue\n            if rel not in artifacts:\n                raise ValueError(f\"Unexpected file not listed in manifest: {rel}\")\n\n    return manifest\n\n\n# ==============================================================================\n# Summary Loading & Checkpoint Resolution\n# ==============================================================================\n\ndef load_experiment_results(summary_path: Path | str) -> Dict[str, Any]:\n    \"\"\"Load experiment_results.json, validating completion and refusing smoke summaries.\"\"\"\n    p = Path(summary_path).resolve()\n    if p.is_symlink():\n        raise PathTraversalError(f\"Symlink rejected for summary file: {p}\")\n    if not p.exists():\n        raise FileNotFoundError(f\"experiment_results.json not found at: {p}\")\n\n    with open(p, \"r\", encoding=\"utf-8\") as f:\n        summary = json.load(f)\n\n    if summary.get(\"status\") != \"COMPLETED\":\n        raise ValueError(f\"Experiment status is '{summary.get('status')}', not COMPLETED.\")\n\n    if summary.get(\"smoke_synthetic\") is True or summary.get(\"execution_mode\") == \"SMOKE_SYNTHETIC\":\n        raise SmokeExperimentError(\n            \"Refusing smoke experiment: evaluation-only runner requires real trained checkpoints.\"\n        )\n\n    if \"arms\" not in summary:\n        raise ValueError(\"Missing 'arms' section in experiment_results.json.\")\n\n    return summary\n\n\ndef resolve_arm_checkpoint_target(\n    arm_name: str,\n    arm_summary: Dict[str, Any],\n    experiment_dir: Path,\n    base_model_dir: Path,\n) -> Tuple[int, Path, str]:\n    \"\"\"Resolve (selected_step, resolved_path, source_desc) for an arm with strict cross-checks.\n\n    Rules:\n    - Arm_A: must be step 0, resolves to base bundle model\n    - Arm_B / Arm_C:\n        * selected_step must be an integer >= 0\n        * cross-checked against val_history entries (step, checkpoint_path)\n        * if selected_step == 0: must have checkpoint_path == null or base model; rejects non-zero positive steps claiming step 0\n        * if selected_step > 0: rejects null checkpoint_path, must match checkpoints/{arm_name}/step_{selected_step}\n        * Kaggle absolute path relocation validated strictly relative to experiment_dir\n        * Fails closed on path traversal, missing checkpoints (no fallback), symlinks, or cross-check discrepancy\n    \"\"\"\n    base_model_resolved = base_model_dir.resolve()\n    if base_model_resolved.is_symlink():\n        raise PathTraversalError(f\"Base model dir is a symlink: {base_model_resolved}\")\n\n    if arm_name == \"Arm_A\":\n        selected_step = arm_summary.get(\"selected_step\", 0)\n        if selected_step != 0:\n            raise CheckpointValidationError(f\"Arm_A selected_step must be 0, got {selected_step}\")\n        return 0, base_model_resolved, \"base_bundle_model_step0\"\n\n    selected_step = arm_summary.get(\"selected_step\")\n    if selected_step is None or not isinstance(selected_step, int):\n        raise CheckpointValidationError(f\"Missing or non-integer selected_step in summary for {arm_name}: {selected_step}\")\n\n    val_history = arm_summary.get(\"val_history\", [])\n    if not val_history:\n        raise CheckpointValidationError(f\"Missing val_history for {arm_name}\")\n\n    # Cross-check selected_step in val_history\n    history_steps = {h.get(\"step\"): h for h in val_history if isinstance(h, dict) and \"step\" in h}\n    if selected_step not in history_steps:\n        raise CheckpointValidationError(\n            f\"Selected step {selected_step} for {arm_name} does not exist in val_history: {list(history_steps.keys())}\"\n        )\n\n    matched_history_row = history_steps[selected_step]\n\n    if selected_step == 0:\n        # Must be baseline step 0\n        ckpt_path_val = arm_summary.get(\"checkpoint_path\")\n        if ckpt_path_val is not None and str(ckpt_path_val).strip() != \"\":\n            # If checkpoint_path is present for step 0, ensure it does not point to a trained non-zero checkpoint dir\n            if f\"step_\" in str(ckpt_path_val) and not str(ckpt_path_val).endswith(\"step_0\"):\n                raise CheckpointValidationError(\n                    f\"Arm {arm_name} selected_step is 0 but checkpoint_path points to positive step: {ckpt_path_val}\"\n                )\n        return 0, base_model_resolved, f\"{arm_name}_step0_base_bundle_model\"\n\n    # For selected_step > 0:\n    exp_dir_resolved = experiment_dir.resolve()\n    if exp_dir_resolved.is_symlink():\n        raise PathTraversalError(f\"Experiment dir is a symlink: {exp_dir_resolved}\")\n\n    # Standard expected relative checkpoint directory layout: checkpoints/{arm_name}/step_{selected_step}\n    expected_rel_path = Path(\"checkpoints\") / arm_name / f\"step_{selected_step}\"\n    expected_full_path = (exp_dir_resolved / expected_rel_path).resolve()\n\n    # Inspect summary checkpoint_path\n    ckpt_path_str = arm_summary.get(\"checkpoint_path\")\n    if not ckpt_path_str:\n        raise CheckpointValidationError(\n            f\"Arm {arm_name} has positive selected_step={selected_step} but missing checkpoint_path in summary\"\n        )\n\n    # Relocate recorded path safely to exp_dir_resolved\n    p = Path(ckpt_path_str)\n    candidate_paths: List[Path] = [expected_full_path]\n\n    if p.is_absolute():\n        parts = p.parts\n        if \"checkpoints\" in parts:\n            idx = parts.index(\"checkpoints\")\n            subpath = Path(*parts[idx:])\n            candidate_paths.insert(0, (exp_dir_resolved / subpath).resolve())\n        elif \"experiment_outputs\" in parts:\n            idx = parts.index(\"experiment_outputs\")\n            subpath = Path(*parts[idx + 1:])\n            candidate_paths.insert(0, (exp_dir_resolved / subpath).resolve())\n        else:\n            # Absolute path does not match expected experiment layout -> path traversal or outside exp_dir\n            raise PathTraversalError(\n                f\"Path traversal or invalid checkpoint location for {arm_name}: {p}\"\n            )\n    else:\n        candidate_paths.insert(0, (exp_dir_resolved / p).resolve())\n\n    # Path traversal validation\n    resolved_ckpt = None\n    for cand in candidate_paths:\n        try:\n            cand.relative_to(exp_dir_resolved)\n        except ValueError:\n            raise PathTraversalError(\n                f\"Path traversal or invalid checkpoint location for {arm_name}: {cand} outside {exp_dir_resolved}\"\n            )\n\n        # Enforce that checkpoint path matches expected step and arm\n        if cand.name != f\"step_{selected_step}\" or cand.parent.name != arm_name:\n            raise CheckpointValidationError(\n                f\"Checkpoint path '{cand}' violates expected layout checkpoints/{arm_name}/step_{selected_step}\"\n            )\n\n        if cand.is_symlink():\n            raise PathTraversalError(f\"Checkpoint directory is a symlink: {cand}\")\n\n        if cand.exists() and cand.is_dir():\n            resolved_ckpt = cand\n            break\n\n    if resolved_ckpt is None:\n        raise MissingCheckpointError(\n            f\"Checkpoint directory missing on disk for {arm_name} step {selected_step}: {candidate_paths[0]}\"\n        )\n\n    # Cross-check with val_history checkpoint_path if recorded\n    val_ckpt = matched_history_row.get(\"checkpoint_path\")\n    if val_ckpt:\n        v_name = Path(val_ckpt).name\n        if v_name != f\"step_{selected_step}\":\n            raise CheckpointValidationError(\n                f\"val_history row for step {selected_step} records mismatched checkpoint {val_ckpt}\"\n            )\n\n    return selected_step, resolved_ckpt, f\"checkpoint_step_{selected_step}\"\n\n\n# ==============================================================================\n# Model Loading & Embedding Wrapper (Eval-Only)\n# ==============================================================================\n\nclass EvalModelWrapper:\n    \"\"\"Read-only inference wrapper ensuring model remains in eval mode with no parameter updates.\"\"\"\n    def __init__(self, model: Any, tokenizer: Any, device: Any):\n        self.model = model\n        self.tokenizer = tokenizer\n        self.device = device\n        self.model.eval()\n\n    def encode(self, texts: List[str], batch_size: int = 32) -> List[List[float]]:\n        import torch\n        import torch.nn.functional as F\n\n        self.model.eval()\n        all_embs = []\n        with torch.no_grad():\n            for i in range(0, len(texts), batch_size):\n                chunk = texts[i:i + batch_size]\n                batch = self.tokenizer(\n                    chunk,\n                    padding=True,\n                    truncation=True,\n                    max_length=256,\n                    return_tensors=\"pt\",\n                )\n                input_ids = batch[\"input_ids\"].to(self.device)\n                attention_mask = batch[\"attention_mask\"].to(self.device)\n                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)\n                token_embeddings = outputs[0]\n                input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()\n                sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)\n                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)\n                pooled = sum_embeddings / sum_mask\n                normalized = F.normalize(pooled, p=2, dim=1)\n                all_embs.extend(normalized.cpu().tolist())\n        return all_embs\n\n\ndef load_model_for_arm(model_dir: Path | str, device: str = \"cpu\") -> Any:\n    \"\"\"Load model and tokenizer strictly in eval mode with no gradients.\n\n    Requires use_safetensors=True without fallback to arbitrary unpickling.\n    \"\"\"\n    p = Path(model_dir).resolve()\n    if p.is_symlink():\n        raise PathTraversalError(f\"Model directory is a symlink: {p}\")\n    dir_str = str(p)\n\n    try:\n        from transformers import AutoModel, AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(dir_str, local_files_only=True)\n        # Strict use_safetensors=True - no insecure pickle fallback!\n        model = AutoModel.from_pretrained(dir_str, local_files_only=True, use_safetensors=True).to(device)\n        for param in model.parameters():\n            param.requires_grad_(False)\n        return EvalModelWrapper(model=model, tokenizer=tokenizer, device=device)\n    except Exception as e:\n        # Fall back only to sentence-transformers if safetensors model load via AutoModel fails\n        from sentence_transformers import SentenceTransformer\n        st = SentenceTransformer(dir_str, device=str(device), model_kwargs={\"local_files_only\": True})\n        st.eval()\n        for param in st.parameters():\n            param.requires_grad_(False)\n\n        class STEvalAdapter:\n            def __init__(self, st_model, dev):\n                self.st = st_model\n                self.device = dev\n                self.model = st_model\n\n            def encode(self, texts: List[str], batch_size: int = 32) -> List[List[float]]:\n                self.st.eval()\n                res = self.st.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)\n                return res.tolist() if hasattr(res, \"tolist\") else res\n\n        return STEvalAdapter(st, device)\n\n\n# ==============================================================================\n# Candidate Pool Builder with Evidence Identity Disambiguation\n# ==============================================================================\n\ndef normalize_semantic_text(value: Optional[str]) -> str:\n    \"\"\"Normalize text matching standard repo convention: NFKC, lowercase, single-spaced.\"\"\"\n    if not value:\n        return \"\"\n    return re.sub(r\"\\s+\", \" \", unicodedata.normalize(\"NFKC\", str(value))).strip().lower()\n\n\ndef build_eval_pools_with_unique_ids(\n    records: List[Dict[str, Any]],\n    split: str = \"test\",\n) -> List[Dict[str, Any]]:\n    \"\"\"Group split records by (job_id, requirement_id, requirement), ensuring unique candidate identity.\n\n    Candidate_id alone has duplicates (e.g. 10 occurrences); unique evidence key is constructed\n    as candidate_id::evidence_id::record_id to prevent collision or misattribution.\n    \"\"\"\n    grouped: Dict[Tuple[str, str, str], List[Dict[str, Any]]] = defaultdict(list)\n    for r in records:\n        if r.get(\"split\") == split:\n            key = (\n                str(r[\"job_id\"]),\n                str(r[\"requirement_id\"]),\n                str(r.get(\"requirement\", \"\")),\n            )\n            grouped[key].append(r)\n\n    pools = []\n    for (job_id, req_id, req_text), row_list in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):\n        norm_seen: Dict[str, int] = {}\n        deduped_candidates = []\n\n        for row in row_list:\n            evi_norm = normalize_semantic_text(row.get(\"evidence\", \"\"))\n            lbl = int(row.get(\"label\", 0))\n\n            if evi_norm in norm_seen:\n                prev_lbl = norm_seen[evi_norm]\n                if prev_lbl != lbl:\n                    raise ValueError(\n                        f\"Conflicting labels for duplicate evidence in query ({job_id}, {req_id}): \"\n                        f\"existing {prev_lbl} vs new {lbl}\"\n                    )\n                continue\n\n            norm_seen[evi_norm] = lbl\n            cand_id = str(row.get(\"candidate_id\") or \"\")\n            ev_id = str(row.get(\"evidence_id\") or \"\")\n            rec_id = str(row.get(\"record_id\") or \"\")\n            unique_key = f\"{cand_id}::{ev_id}::{rec_id}\" if (ev_id or rec_id) else cand_id\n\n            evidence_text = str(row.get(\"evidence\", \"\"))\n            text_sha = hashlib.sha256(evidence_text.encode(\"utf-8\")).hexdigest()\n\n            deduped_candidates.append({\n                \"unique_evidence_key\": unique_key,\n                \"candidate_id\": cand_id,\n                \"evidence_id\": ev_id,\n                \"record_id\": rec_id,\n                \"evidence\": evidence_text,\n                \"normalized_evidence\": evi_norm,\n                \"evidence_sha256\": text_sha,\n                \"label\": lbl,\n            })\n\n        pos_count = sum(1 for c in deduped_candidates if c[\"label\"] == 1)\n        lang = req_id.split(\"@\")[-1] if \"@\" in req_id else \"unknown\"\n\n        pools.append({\n            \"split\": split,\n            \"job_id\": job_id,\n            \"requirement_id\": req_id,\n            \"requirement\": req_text,\n            \"requirement_sha256\": hashlib.sha256(req_text.encode(\"utf-8\")).hexdigest(),\n            \"language\": lang,\n            \"candidates\": deduped_candidates,\n            \"pool_size\": len(deduped_candidates),\n            \"pos_count\": pos_count,\n            \"has_positive\": pos_count > 0,\n        })\n\n    return pools\n\n\n# ==============================================================================\n# Model Evaluation & Metrics (All, Mixed-Only, Positive-Only, Pairwise)\n# ==============================================================================\n\ndef cosine_similarity(v1: List[float], v2: List[float]) -> float:\n    score = sum(a * b for a, b in zip(v1, v2))\n    if not math.isfinite(score):\n        raise NonFiniteScoreError(f\"Non-finite cosine score detected: {score}\")\n    return float(score)\n\n\ndef evaluate_arm_model(\n    embedder: Any,\n    pools: List[Dict[str, Any]],\n    arm_name: str,\n) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:\n    \"\"\"Score candidate pools with embedder, sorting deterministically with tie tracking.\"\"\"\n    unique_texts = set()\n    for p in pools:\n        unique_texts.add(p[\"requirement\"])\n        for c in p[\"candidates\"]:\n            unique_texts.add(c[\"evidence\"])\n\n    text_list = sorted(unique_texts)\n    emb_list = embedder.encode(text_list)\n    emb_map = {t: emb for t, emb in zip(text_list, emb_list)}\n\n    scored_cases = []\n    for p in pools:\n        q_emb = emb_map[p[\"requirement\"]]\n        scored_cands = []\n        for c in p[\"candidates\"]:\n            e_emb = emb_map[c[\"evidence\"]]\n            score = cosine_similarity(q_emb, e_emb)\n            scored_cands.append({\n                \"unique_evidence_key\": c[\"unique_evidence_key\"],\n                \"candidate_id\": c[\"candidate_id\"],\n                \"evidence_id\": c[\"evidence_id\"],\n                \"record_id\": c[\"record_id\"],\n                \"evidence_sha256\": c[\"evidence_sha256\"],\n                \"label\": c[\"label\"],\n                \"score\": score,\n            })\n\n        # Deterministic tie-breaking: sort descending by score, then ascending by unique_evidence_key\n        scored_cands.sort(key=lambda x: (-x[\"score\"], x[\"unique_evidence_key\"]))\n\n        scored_cases.append({\n            \"split\": p[\"split\"],\n            \"job_id\": p[\"job_id\"],\n            \"requirement_id\": p[\"requirement_id\"],\n            \"requirement\": p[\"requirement\"],\n            \"requirement_sha256\": p[\"requirement_sha256\"],\n            \"language\": p[\"language\"],\n            \"candidates\": scored_cands,\n        })\n\n    metrics = compute_diagnostic_metrics(scored_cases)\n    return metrics, scored_cases\n\n\ndef compute_diagnostic_metrics(scored_cases: List[Dict[str, Any]]) -> Dict[str, Any]:\n    \"\"\"Compute micro and macro_job metrics across partitions with pairwise tie accounting.\"\"\"\n    all_positive_cases = []\n    mixed_cases = []\n    positive_only_cases = []\n    no_positive_cases = []\n\n    pos_neg_pairs_total = 0\n    pos_beats_neg_total = 0\n    pos_loses_neg_total = 0\n    pos_neg_ties_total = 0\n    total_score_ties = 0\n\n    for case in scored_cases:\n        cands = case[\"candidates\"]\n        scores = [c[\"score\"] for c in cands]\n        for s in scores:\n            if not math.isfinite(s):\n                raise NonFiniteScoreError(f\"Non-finite cosine score detected: {s}\")\n\n        # Count ties among all candidate scores in this pool\n        for i in range(len(scores)):\n            for j in range(i + 1, len(scores)):\n                if abs(scores[i] - scores[j]) < 1e-12:\n                    total_score_ties += 1\n\n        pos_cands = [c for c in cands if c[\"label\"] == 1]\n        neg_cands = [c for c in cands if c[\"label\"] == 0]\n\n        # Pairwise positive vs negative comparisons\n        for pos in pos_cands:\n            for neg in neg_cands:\n                pos_neg_pairs_total += 1\n                if pos[\"score\"] > neg[\"score\"] + 1e-12:\n                    pos_beats_neg_total += 1\n                elif neg[\"score\"] > pos[\"score\"] + 1e-12:\n                    pos_loses_neg_total += 1\n                else:\n                    pos_neg_ties_total += 1\n\n        total_positives = len(pos_cands)\n        pool_size = len(cands)\n\n        if total_positives == 0:\n            m = {\n                \"has_positive\": False,\n                \"total_positives\": 0,\n                \"pool_size\": pool_size,\n                \"mrr\": None,\n                \"hit1\": None,\n                \"hit3\": None,\n                \"recall1\": None,\n                \"recall3\": None,\n            }\n            c_copy = dict(case)\n            c_copy[\"metrics\"] = m\n            no_positive_cases.append(c_copy)\n            continue\n\n        ranks = [idx + 1 for idx, c in enumerate(cands) if c[\"label\"] == 1]\n        first_rank = ranks[0]\n        mrr = 1.0 / first_rank\n        hit1 = 1.0 if first_rank <= 1 else 0.0\n        hit3 = 1.0 if first_rank <= 3 else 0.0\n        pos_in_top1 = sum(1 for r in ranks if r <= 1)\n        pos_in_top3 = sum(1 for r in ranks if r <= 3)\n        recall1 = pos_in_top1 / total_positives\n        recall3 = pos_in_top3 / total_positives\n\n        m = {\n            \"has_positive\": True,\n            \"total_positives\": total_positives,\n            \"pool_size\": pool_size,\n            \"mrr\": mrr,\n            \"hit1\": hit1,\n            \"hit3\": hit3,\n            \"recall1\": recall1,\n            \"recall3\": recall3,\n        }\n        c_copy = dict(case)\n        c_copy[\"metrics\"] = m\n        all_positive_cases.append(c_copy)\n\n        if len(neg_cands) > 0:\n            mixed_cases.append(c_copy)\n        else:\n            positive_only_cases.append(c_copy)\n\n    def aggregate_subset(eval_cases: List[Dict[str, Any]]) -> Dict[str, Any]:\n        metric_keys = [\"mrr\", \"hit1\", \"hit3\", \"recall1\", \"recall3\"]\n        n_queries = len(eval_cases)\n        if n_queries == 0:\n            return {\n                \"queries\": 0,\n                \"jobs\": 0,\n                \"micro\": {k: None for k in metric_keys},\n                \"macro_job\": {k: None for k in metric_keys},\n                \"job_averages\": {},\n            }\n\n        micro = {k: sum(c[\"metrics\"][k] for c in eval_cases) / n_queries for k in metric_keys}\n\n        by_job: Dict[str, List[Dict[str, Any]]] = defaultdict(list)\n        for c in eval_cases:\n            by_job[c[\"job_id\"]].append(c[\"metrics\"])\n\n        job_averages: Dict[str, Dict[str, float]] = {}\n        for jid, q_list in by_job.items():\n            job_averages[jid] = {k: sum(q[k] for q in q_list) / len(q_list) for k in metric_keys}\n\n        job_count = len(job_averages)\n        macro_job = {k: sum(job_averages[j][k] for j in job_averages) / job_count for k in metric_keys}\n        macro_job[\"job_count\"] = job_count\n\n        return {\n            \"queries\": n_queries,\n            \"jobs\": job_count,\n            \"micro\": micro,\n            \"macro_job\": macro_job,\n            \"job_averages\": job_averages,\n        }\n\n    return {\n        \"all_positive\": aggregate_subset(all_positive_cases),\n        \"mixed_only\": aggregate_subset(mixed_cases),\n        \"positive_only\": aggregate_subset(positive_only_cases),\n        \"no_positive\": {\n            \"queries\": len(no_positive_cases),\n            \"jobs\": len(set(c[\"job_id\"] for c in no_positive_cases)),\n        },\n        \"pairwise_ordering\": {\n            \"pos_neg_pairs\": pos_neg_pairs_total,\n            \"pos_beats_neg\": pos_beats_neg_total,\n            \"pos_loses_neg\": pos_loses_neg_total,\n            \"pos_neg_ties\": pos_neg_ties_total,\n            \"total_score_ties\": total_score_ties,\n        }\n    }\n\n\ndef paired_job_bootstrap_eval(\n    job_scores_primary: Dict[str, float],\n    job_scores_baseline: Dict[str, float],\n    n_resamples: int = 1000,\n    seed: int = 42,\n) -> Dict[str, Any]:\n    \"\"\"Seeded paired bootstrap for job-macro MRR difference.\"\"\"\n    common_jobs = sorted(set(job_scores_primary.keys()) & set(job_scores_baseline.keys()))\n    if not common_jobs:\n        return {\n            \"mean_delta\": 0.0,\n            \"ci_95_low\": 0.0,\n            \"ci_95_high\": 0.0,\n            \"common_jobs_count\": 0,\n            \"resamples\": n_resamples,\n            \"seed\": seed,\n        }\n\n    deltas = [job_scores_primary[j] - job_scores_baseline[j] for j in common_jobs]\n    n_jobs = len(deltas)\n    mean_delta = sum(deltas) / n_jobs\n\n    rng = random.Random(seed)\n    boot_means = []\n    for _ in range(n_resamples):\n        sample = [rng.choice(deltas) for _ in range(n_jobs)]\n        boot_means.append(sum(sample) / n_jobs)\n\n    boot_means.sort()\n    low_idx = int(0.025 * n_resamples)\n    high_idx = int(0.975 * n_resamples)\n\n    return {\n        \"mean_delta\": mean_delta,\n        \"ci_95_low\": boot_means[low_idx],\n        \"ci_95_high\": boot_means[high_idx],\n        \"common_jobs_count\": n_jobs,\n        \"resamples\": n_resamples,\n        \"seed\": seed,\n    }\n\n\ndef build_per_query_evidence_comparison(\n    cases_primary: List[Dict[str, Any]],\n    cases_baseline: List[Dict[str, Any]],\n    primary_name: str,\n    baseline_name: str,\n) -> List[Dict[str, Any]]:\n    \"\"\"Compare candidate score and rank changes matched strictly by unique_evidence_key.\"\"\"\n    baseline_map: Dict[Tuple[str, str, str], Dict[str, Any]] = {}\n    for case in cases_baseline:\n        jid = case[\"job_id\"]\n        rid = case[\"requirement_id\"]\n        for rank_idx, c in enumerate(case[\"candidates\"], start=1):\n            key = (jid, rid, c[\"unique_evidence_key\"])\n            baseline_map[key] = {\n                \"rank\": rank_idx,\n                \"score\": c[\"score\"],\n            }\n\n    comparisons = []\n    for case in cases_primary:\n        jid = case[\"job_id\"]\n        rid = case[\"requirement_id\"]\n        for rank_idx, c in enumerate(case[\"candidates\"], start=1):\n            key = (jid, rid, c[\"unique_evidence_key\"])\n            base_info = baseline_map.get(key)\n            if base_info is None:\n                continue\n\n            score_delta = c[\"score\"] - base_info[\"score\"]\n            rank_delta = base_info[\"rank\"] - rank_idx  # Positive means rank improved\n\n            comparisons.append({\n                \"job_id\": jid,\n                \"requirement_id\": rid,\n                \"unique_evidence_key\": c[\"unique_evidence_key\"],\n                \"candidate_id\": c[\"candidate_id\"],\n                \"evidence_id\": c[\"evidence_id\"],\n                \"record_id\": c[\"record_id\"],\n                \"evidence_sha256\": c[\"evidence_sha256\"],\n                \"label\": c[\"label\"],\n                f\"{primary_name}_score\": c[\"score\"],\n                f\"{primary_name}_rank\": rank_idx,\n                f\"{baseline_name}_score\": base_info[\"score\"],\n                f\"{baseline_name}_rank\": base_info[\"rank\"],\n                \"score_delta\": score_delta,\n                \"rank_delta\": rank_delta,\n            })\n\n    return comparisons\n\n\n# ==============================================================================\n# Main Orchestrator\n# ==============================================================================\n\ndef run_eval_only(\n    bundle_dir: Path | str,\n    experiment_dir: Path | str,\n    output_dir: Path | str,\n    device: str = \"cpu\",\n    split: str = \"test\",\n) -> Dict[str, Any]:\n    \"\"\"Execute evaluation-only pipeline across Arms A, B, and C.\"\"\"\n    b_dir = Path(bundle_dir).resolve()\n    e_dir = Path(experiment_dir).resolve()\n    out_dir = Path(output_dir).resolve()\n\n    if b_dir.is_symlink():\n        raise PathTraversalError(f\"Bundle dir cannot be a symlink: {b_dir}\")\n    if e_dir.is_symlink():\n        raise PathTraversalError(f\"Experiment dir cannot be a symlink: {e_dir}\")\n    if out_dir.is_symlink():\n        raise PathTraversalError(f\"Output dir cannot be a symlink: {out_dir}\")\n\n    # Fail closed on output directory collision\n    if out_dir.exists() and any(out_dir.iterdir()):\n        raise RefuseOverwriteError(\n            f\"Collision: output directory exists and is not empty: {out_dir}\"\n        )\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    # 1. Verify bundle integrity\n    print(f\"[1/6] Verifying bundle manifest at {b_dir}...\")\n    manifest = verify_bundle_manifest(b_dir)\n\n    # 2. Load summary and check for smoke run\n    summary_path = e_dir / \"experiment_results.json\"\n    print(f\"[2/6] Loading experiment results from {summary_path}...\")\n    summary = load_experiment_results(summary_path)\n\n    # 3. Resolve checkpoints for Arms A, B, C\n    print(\"[3/6] Resolving checkpoint targets (no training / no selection mutation)...\")\n    base_model_dir = b_dir / \"model\"\n    arm_targets = {}\n    for arm_name in [\"Arm_A\", \"Arm_B\", \"Arm_C\"]:\n        arm_summary = summary[\"arms\"].get(arm_name, {})\n        step, ckpt_path, desc = resolve_arm_checkpoint_target(\n            arm_name, arm_summary, e_dir, base_model_dir\n        )\n        arm_targets[arm_name] = {\n            \"selected_step\": step,\n            \"checkpoint_path\": str(ckpt_path),\n            \"description\": desc,\n            \"checkpoint_files_sha256\": directory_sha256_manifest(ckpt_path),\n        }\n        print(f\"  - {arm_name}: step {step} at {ckpt_path}\")\n\n    # 4. Load semantic test records with unique evidence IDs\n    print(f\"[4/6] Building restricted pools for split='{split}' with unique evidence keys...\")\n    semantic_path = b_dir / \"data\" / \"semantic.jsonl\"\n    records = []\n    with open(semantic_path, \"r\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                records.append(json.loads(line))\n\n    pools = build_eval_pools_with_unique_ids(records, split=split)\n    print(f\"  - Loaded {len(pools)} {split} queries across {len(set(p['job_id'] for p in pools))} jobs.\")\n\n    # 5. Evaluate Arms A, B, C\n    print(\"[5/6] Evaluating models strictly in eval mode...\")\n    arm_eval_results = {}\n    arm_scored_cases = {}\n    for arm_name, target in arm_targets.items():\n        print(f\"  Evaluating {arm_name} (step {target['selected_step']})...\")\n        model = load_model_for_arm(target[\"checkpoint_path\"], device=device)\n        metrics, scored_cases = evaluate_arm_model(model, pools, arm_name)\n        arm_eval_results[arm_name] = metrics\n        arm_scored_cases[arm_name] = scored_cases\n        del model\n\n    # 6. Bootstrap comparisons & Pairwise rank/score changes\n    print(\"[6/6] Computing paired job bootstrap and matched evidence comparisons...\")\n    job_scores_a = arm_eval_results[\"Arm_A\"][\"all_positive\"][\"job_averages\"]\n    job_scores_b = arm_eval_results[\"Arm_B\"][\"all_positive\"][\"job_averages\"]\n    job_scores_c = arm_eval_results[\"Arm_C\"][\"all_positive\"][\"job_averages\"]\n\n    job_mrr_a = {j: m[\"mrr\"] for j, m in job_scores_a.items()}\n    job_mrr_b = {j: m[\"mrr\"] for j, m in job_scores_b.items()}\n    job_mrr_c = {j: m[\"mrr\"] for j, m in job_scores_c.items()}\n\n    boot_c_a = paired_job_bootstrap_eval(job_mrr_c, job_mrr_a, n_resamples=1000, seed=42)\n    boot_c_b = paired_job_bootstrap_eval(job_mrr_c, job_mrr_b, n_resamples=1000, seed=42)\n\n    comp_c_a = build_per_query_evidence_comparison(arm_scored_cases[\"Arm_C\"], arm_scored_cases[\"Arm_A\"], \"Arm_C\", \"Arm_A\")\n    comp_c_b = build_per_query_evidence_comparison(arm_scored_cases[\"Arm_C\"], arm_scored_cases[\"Arm_B\"], \"Arm_C\", \"Arm_B\")\n\n    # Record environment and provenance\n    try:\n        import torch\n        torch_ver = torch.__version__\n    except ImportError:\n        torch_ver = \"unavailable\"\n\n    try:\n        import transformers\n        trans_ver = transformers.__version__\n    except ImportError:\n        trans_ver = \"unavailable\"\n\n    current_script_path = Path(__file__).resolve()\n    provenance = {\n        \"script\": str(current_script_path),\n        \"script_sha256\": compute_sha256(current_script_path),\n        \"bundle_manifest_sha256\": compute_sha256(b_dir / \"manifest.json\"),\n        \"semantic_data_sha256\": compute_sha256(semantic_path),\n        \"source_experiment_results_sha256\": compute_sha256(summary_path),\n        \"python_version\": sys.version,\n        \"platform\": platform.platform(),\n        \"torch_version\": torch_ver,\n        \"transformers_version\": trans_ver,\n        \"device\": device,\n    }\n\n    final_report = {\n        \"status\": \"COMPLETED\",\n        \"evaluation_type\": \"SAVED_CHECKPOINTS_EVAL_ONLY_NO_RETRAINING\",\n        \"diagnostic_disclaimer\": (\n            \"POST-HOC DIAGNOSTICS ONLY. Evaluation was conducted on saved checkpoints without retraining. \"\n            \"All test metrics and score comparisons are diagnostic and must NOT be used for independent checkpoint selection.\"\n        ),\n        \"provenance\": provenance,\n        \"arm_targets\": arm_targets,\n        \"evaluation_metrics\": arm_eval_results,\n        \"bootstrap_comparisons\": {\n            \"delta_c_minus_a\": boot_c_a,\n            \"delta_c_minus_b\": boot_c_b,\n        },\n        \"pairwise_evidence_comparison_summary\": {\n            \"c_vs_a\": {\n                \"total_matched_evidences\": len(comp_c_a),\n                \"rank_improvements\": sum(1 for x in comp_c_a if x[\"rank_delta\"] > 0),\n                \"rank_regressions\": sum(1 for x in comp_c_a if x[\"rank_delta\"] < 0),\n                \"rank_unchanged\": sum(1 for x in comp_c_a if x[\"rank_delta\"] == 0),\n            },\n            \"c_vs_b\": {\n                \"total_matched_evidences\": len(comp_c_b),\n                \"rank_improvements\": sum(1 for x in comp_c_b if x[\"rank_delta\"] > 0),\n                \"rank_regressions\": sum(1 for x in comp_c_b if x[\"rank_delta\"] < 0),\n                \"rank_unchanged\": sum(1 for x in comp_c_b if x[\"rank_delta\"] == 0),\n            }\n        }\n    }\n\n    # Save outputs\n    metrics_file = out_dir / \"eval_diagnostic_results.json\"\n    with open(metrics_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(final_report, f, indent=2, ensure_ascii=False)\n\n    for arm_name in [\"Arm_A\", \"Arm_B\", \"Arm_C\"]:\n        pred_file = out_dir / f\"eval_predictions_{arm_name}.jsonl\"\n        with open(pred_file, \"w\", encoding=\"utf-8\") as f:\n            for case in arm_scored_cases[arm_name]:\n                f.write(json.dumps(case, ensure_ascii=False) + \"\\n\")\n\n    diff_file_ca = out_dir / \"eval_score_rank_diff_c_vs_a.jsonl\"\n    with open(diff_file_ca, \"w\", encoding=\"utf-8\") as f:\n        for item in comp_c_a:\n            f.write(json.dumps(item, ensure_ascii=False) + \"\\n\")\n\n    diff_file_cb = out_dir / \"eval_score_rank_diff_c_vs_b.jsonl\"\n    with open(diff_file_cb, \"w\", encoding=\"utf-8\") as f:\n        for item in comp_c_b:\n            f.write(json.dumps(item, ensure_ascii=False) + \"\\n\")\n\n    print(f\"\\nEvaluation-only export complete! Files saved to: {out_dir}\")\n    return final_report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Evaluate existing AI1 checkpoints without retraining\")\n    parser.add_argument(\"--bundle\", required=True, help=\"Path to input bundle containing manifest.json\")\n    parser.add_argument(\"--experiment\", required=True, help=\"Path to existing experiment_outputs directory or results zip\")\n    parser.add_argument(\"--output\", required=True, help=\"Path to directory for eval-only outputs (must not exist)\")\n    parser.add_argument(\"--device\", default=\"cpu\", help=\"Device to use for eval (cpu or cuda)\")\n    parser.add_argument(\"--split\", default=\"test\", help=\"Dataset split to evaluate\")\n    args = parser.parse_args()\n\n    exp_path = Path(args.experiment).resolve()\n    if exp_path.is_file() and zipfile.is_zipfile(exp_path):\n        # Extract into a clean dedicated directory\n        extract_dir = exp_path.parent / f\"_extracted_{exp_path.stem}\"\n        if extract_dir.exists():\n            import time\n            extract_dir = exp_path.parent / f\"_extracted_{exp_path.stem}_{int(time.time())}\"\n        print(f\"Extracting results archive {exp_path} to {extract_dir}...\")\n        safe_extract_results_archive(exp_path, extract_dir)\n        exp_dir = extract_dir\n    else:\n        exp_dir = exp_path\n\n    run_eval_only(\n        bundle_dir=args.bundle,\n        experiment_dir=exp_dir,\n        output_dir=args.output,\n        device=args.device,\n        split=args.split,\n    )\n\n\nif __name__ == \"__main__\":\n    main()\n"

with open(EVAL_SCRIPT_PATH, 'w', encoding='utf-8') as f:
    f.write(EVAL_SCRIPT_SOURCE)

print(f'Trusted eval helper written to: {EVAL_SCRIPT_PATH}')
h = hashlib.sha256(EVAL_SCRIPT_PATH.read_bytes()).hexdigest()
print(f'SHA-256: {h}')


In [ ]:
# 4. Execute Evaluation-Only Pipeline
import time
EVAL_OUTPUT_DIR = WORKING_DIR / f'eval_only_diagnostics_{int(time.time())}'
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

device_str = 'cpu'
try:
    import torch
    if torch.cuda.is_available():
        device_str = 'cuda'
except ImportError:
    pass

exp_arg = str(EXP_DIR) if EXP_DIR else str(EXP_ZIP)

cmd = [
    sys.executable, '-I', str(EVAL_SCRIPT_PATH),
    '--bundle', str(BUNDLE_DIR),
    '--experiment', exp_arg,
    '--output', str(EVAL_OUTPUT_DIR),
    '--device', device_str,
    '--split', 'test'
]

print('Executing evaluation-only pipeline:')
print(' '.join(cmd))
subprocess.check_call(cmd)
print('\nEvaluation finished successfully!')


In [ ]:
# 5. Display Comprehensive Diagnostic Test Report
with open(EVAL_OUTPUT_DIR / 'eval_diagnostic_results.json', 'r', encoding='utf-8') as f:
    diag_res = json.load(f)

print('=' * 80)
print('DIAGNOSTIC TEST EVALUATION REPORT (POST-HOC DIAGNOSTICS ONLY)')
print('=' * 80)

def fmt(v, spec='.4f'):
    if v is None: return 'N/A'
    try: return format(v, spec)
    except Exception: return str(v)

eval_m = diag_res['evaluation_metrics']
for partition, p_title in [('all_positive', 'ALL POSITIVE QUERIES'), ('mixed_only', 'MIXED-LABEL ONLY QUERIES')]:
    print(f'\n--- {p_title} ---')
    print(f'{"Arm":8} | {"MRR (Micro)":12} | {"MRR (Macro)":12} | {"Hit@1 (Macro)":14} | {"Hit@3 (Macro)":14} | {"Recall@3 (Macro)":16}')
    print('-' * 86)
    for arm in ['Arm_A', 'Arm_B', 'Arm_C']:
        sub = eval_m[arm][partition]
        print(f'{arm:8} | {fmt(sub["micro"]["mrr"]):12} | {fmt(sub["macro_job"]["mrr"]):12} | {fmt(sub["macro_job"]["hit1"]):14} | {fmt(sub["macro_job"]["hit3"]):14} | {fmt(sub["macro_job"]["recall3"]):16}')

print('\n--- PAIRWISE ORDERING (Positive vs Negative) & TIES ---')
for arm in ['Arm_A', 'Arm_B', 'Arm_C']:
    pw = eval_m[arm]['pairwise_ordering']
    print(f'{arm:8}: Pairs={pw["pos_neg_pairs"]}, Pos Beats Neg={pw["pos_beats_neg"]}, Pos Loses Neg={pw["pos_loses_neg"]}, Pos/Neg Ties={pw["pos_neg_ties"]}, Total Score Ties={pw["total_score_ties"]}')

boots = diag_res['bootstrap_comparisons']
print('\n--- PAIRED JOB BOOTSTRAP 95% CI (Job-Macro MRR Delta) ---')
b_ca = boots['delta_c_minus_a']
b_cb = boots['delta_c_minus_b']
print(f'Delta (C - A): {fmt(b_ca["mean_delta"], "+.4f")} (95% CI: [{fmt(b_ca["ci_95_low"], "+.4f")}, {fmt(b_ca["ci_95_high"], "+.4f")}])')
print(f'Delta (C - B): {fmt(b_cb["mean_delta"], "+.4f")} (95% CI: [{fmt(b_cb["ci_95_low"], "+.4f")}, {fmt(b_cb["ci_95_high"], "+.4f")}])')

comp_summary = diag_res['pairwise_evidence_comparison_summary']
print('\n--- MATCHED EVIDENCE RANK CHANGES (C vs A) ---')
print(f'Total matched candidates: {comp_summary["c_vs_a"]["total_matched_evidences"]}')
print(f'Rank improvements: {comp_summary["c_vs_a"]["rank_improvements"]}')
print(f'Rank regressions: {comp_summary["c_vs_a"]["rank_regressions"]}')
print(f'Rank unchanged: {comp_summary["c_vs_a"]["rank_unchanged"]}')


In [ ]:
# 6. Package Diagnostics Results into ZIP
zip_out = WORKING_DIR / 'ai1_eval_only_diagnostics.zip'
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in EVAL_OUTPUT_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(EVAL_OUTPUT_DIR)))
print(f'Packaged diagnostics zip: {zip_out} ({zip_out.stat().st_size} bytes)')
print('You can download ai1_eval_only_diagnostics.zip from Kaggle Output tab.')
